In [1]:
from EGPA import *
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Add the main directory to sys.path
sys.path.append(main_dir)
from OptFunctions.oneDQHarmOscillatorfunctions import *
#from utils import _parallel_apply_along_axis
import time
#import warnings
#warnings.filterwarnings("ignore")

## **Optimización de los parámetros del EGPA, para oscilador armónico**

Primero voy a ver que tal funciona para el caso del **oscilador armónico**. Dado una colección de hiperparámetros la medida de la calidad de la solución la da el RMSE respecto a la solución analítica conocida. Solo está probado con un número de puntos de grid fijo.

In [2]:
import numpy as np
from skopt import gp_minimize
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args

n_variables = [50, 300, 550, 800] #Tamaños del grid (número de dimensiones del problema de optimización)

# Optimización de parámetros de EGPA
# Define the search space
search_space = [
    Categorical(50, 1000, name="size_gen"),          # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
    Categorical(10, 100, name="dead_iter"),          # Discrete values
    Real(0.02, 0.4, prior='log-uniform' ,name="alpha"),               # Continuous values
    Real(0.02, 0.4, prior='log-uniform' ,name="elite_factor"),        # Continuous values
    Real(0.01, 0.2, prior='log-uniform' ,name="std_lower_bound"),     # Continuous values
    Categorical(2, 10, name="init_kernels")          # Discrete values
]

# Simulated function that runs the optimization and computes RMSE
def evaluate_algorithm(size_gen, dead_iter, alpha, elite_factor, std_lower_bound, init_kernels):
    """
    This function should run the actual optimization algorithm using the given parameters.
    It should return the worst RMSE (max RMSE across the three optimization problems).
    
    Replace the `np.random.uniform()` call with the actual function evaluation.
    """
    X_grid = np.linspace(start=-4, stop=4, num=n_variables[0]).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=-4 (-inf) x=4 (inf) que son 0
    X_train = X_grid[[0,-1]]
    Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array

    #Analytical solutions definitions
    #Ground state
    APsi0 = np.exp(-X_grid**2/2)
    integral =0
    for i in range(len(APsi0)):
        integral = integral + (APsi0[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi0)-1))

    B = (1/integral)**0.5
    APsi0 = B * APsi0

    #1st excitated state
    APsi1 = np.exp(-X_grid**2/2)*2*X_grid
    integral =0
    for i in range(len(APsi1)):
        integral = integral + (APsi1[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi1)-1))

    B = (1/integral)**0.5
    APsi1 = -1 * B * APsi1 #signo menos para correspondencia con solución algoritmo (simetrización)

    #2nd excitated state
    APsi2 = np.exp(-X_grid**2/2)*(4*X_grid**2-2)
    integral =0
    for i in range(len(APsi2)):
        integral = integral + (APsi2[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi2)-1))

    B = (1/integral)**0.5
    APsi2 = B * APsi2

    #3rd excitated state
    APsi3 = np.exp(-X_grid**2/2)*(8*X_grid**3-12*X_grid)
    integral =0
    for i in range(len(APsi3)):
        integral = integral + (APsi3[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi3)-1))

    B = (1/integral)**0.5
    APsi3 = -1 * B * APsi3 #signo menos para correspondencia con solución algoritmo (simetrización)

    APsis = {}
    APsis['APsi0'] = APsi0[:, 0]
    APsis['APsi1'] = APsi1[:, 0]
    APsis['APsi2'] = APsi2[:, 0]
    APsis['APsi3'] = APsi3[:, 0]

    ### Ahora el código con EGPA que calcula los 4 primeros estados excitados del problema oscilador armónico
    params = {'x':X_grid}
    #Ground state
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=350, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHOground = Gaussian.minimize(oneDHOExpE, params, exp_from = 3, output_runtime=False)
    #Se guardan resultados de algoritmo para siguiente estado excitado
    Psi_ground=egparesultQHOground.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E0 = np.sqrt(-np.log(-egparesultQHOground.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así 
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground}
    #print(f'  |Psi0> done\n')

    #1er estado excitado
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=350, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHO1 = Gaussian.minimize(oneDHOExpE1, params, exp_from = 5, output_runtime=False)
    Psi1=egparesultQHO1.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E1 = np.sqrt(-np.log(-egparesultQHO1.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1}
    #print(f'  |Psi1> done\n')

    #2o estado excitado
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=350, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHO2 = Gaussian.minimize(oneDHOExpE2, params, exp_from = 5, output_runtime=False)
    Psi2=egparesultQHO2.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E2 = np.sqrt(-np.log(-egparesultQHO2.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2 }
    #print(f'  |Psi2> done\n')

    #3er estado excitado
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=350, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHO3 = Gaussian.minimize(oneDHOExpE3, params, exp_from = 5, output_runtime=False)
    Psi3=egparesultQHO3.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E3 = np.sqrt(-np.log(-egparesultQHO3.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2, 'E3': E3, 'Psi3': Psi3 }
    #print(f'  |Psi3> done\n')

    #Cálculos de métricas error RMSE de cada estado con estado teórico
    MSEs = []
    for i in range(4):
        mse = np.mean((params[f'Psi{i}'] - APsis[f'APsi{i}'])**2)
        MSEs.append(mse)

    RMSE_global = np.sum(MSEs)
    ######################################################################################
    ######################################################################################
    #rmse_values = np.random.uniform(0.1, 1.0, 3)  # Replace with real RMSE computations
    return RMSE_global #max(rmse_values)  # Min-Max approach: minimize worst RMSE

# Wrapper function for Bayesian Optimization
@use_named_args(search_space)
def objective(**params):
    return evaluate_algorithm(**params)

# Run Bayesian Optimization
result = gp_minimize(
    func=objective,      # Objective function
    dimensions=search_space,  # Search space
    n_calls=50,         # Number of function evaluations
    random_state=42     # For reproducibility
)

# Best parameters found
best_params = dict(zip(["size_gen", "dead_iter", "alpha", "elite_factor", "std_lower_bound", "init_kernels"], result.x))
print("Best Parameters:", best_params)
print("Best RMSE (Min-Max Approach):", result.fun)


TypeError: 'int' object is not iterable

## **Optimización de parámetros de EGPA para los tres problemas de optimización 1D a la vez**
La medida de la calidad del algoritmo para unos hiperparámetros dados la da el máximo entre [media de MSEs para los cuatro primeros niveles del Oscilador armónico, media MSE 4 niveles potencial de caja, MSE en el problema NLSE]. Para optimizar los parámetros se intenta mejorar el peor resultado $$min(max(MSE(4lvlsQOSc) + MSE(4lvlsWell) + MSE(NLSE)))$$

In [ ]:
# from EGPA import *
# import numpy as np
# import sys
# import os
# # Get the absolute path of the main directory
# main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# # Add the main directory to sys.path
# sys.path.append(main_dir)
# from OptFunctions.oneDQHarmOscillatorfunctions import *
# from OptFunctions.NonLinearSE import *
# from OptFunctions.oneDwellfunctions import *
# #from utils import _parallel_apply_along_axis
# import time
# import warnings
# warnings.filterwarnings("ignore")

# #np.int = int
# from skopt import gp_minimize
# from skopt.space import Integer, Real, Categorical
# from skopt.utils import use_named_args

# n_variables = [50, 300, 550, 800] #Tamaños del grid (número de dimensiones del problema de optimización)

# # Optimización de parámetros de EGPA
# # Define the search space
# #¿¿?¿?¿?¿?¿?¿?¿?¿? Categorical([10, 30, 60, 100], name="dead_iter") does not work becaues GP cannot handle categorical values
# # search_space = [
# #     Integer(50, 1000, name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
# #     Integer(10, 100, name="dead_iter"),   # Discrete values Categorical([10, 30, 60, 100], name="dead_iter") was not working because GP cannot handle categorical!!!!
# #     Real(0.02, 0.4, name="alpha", prior='log-uniform'),                # Continuous values
# #     Real(0.02, 0.4, name="elite_factor", prior='log-uniform'),        # Continuous values
# #     Real(0.01, 0.2, name="std_lower_bound", prior='log-uniform'),     # Continuous values
# #     Integer(2, 10, name="init_kernels")      # Discrete values
# # ]

# search_space = [
#     Categorical([50, 150, 350, 550, 850, 1000], name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
#     Categorical([10, 30, 60, 100], name="dead_iter"),   # Discrete values Categorical([10, 30, 60, 100], name="dead_iter") was not working because GP cannot handle categorical!!!!
#     Real(0.02, 0.4, name="alpha", prior='log-uniform'),                # Continuous values
#     Real(0.02, 0.4, name="elite_factor", prior='log-uniform'),        # Continuous values
#     Real(0.01, 0.2, name="std_lower_bound", prior='log-uniform'),     # Continuous values
#     Integer(2, 10, name="init_kernels")      # Discrete values
# ]

# global timesQOSc
# timesQOsc = 0
# global timesWell
# timesWell = 0
# global timesNLSE
# timesNLSE = 0

# # Simulated function that runs the optimization and computes RMSE
# def evaluate_algorithm(size_gen, dead_iter, alpha, elite_factor, std_lower_bound, init_kernels):
#     """
#     This function should run the actual optimization algorithm using the given parameters.
#     It should return the worst MSE (max MSE across the three optimization problems).
    
#     Replace the `np.random.uniform()` call with the actual function evaluation.
#     """
#     ##########################################################################################
#     ##########################################################################################
#     ########################### OSCILADOR ARMÓNICO ###########################################
#     ##########################################################################################
#     print('Quantum Harmonic Oscillator IN PROGRESS\n')
#     X_grid = np.linspace(start=-4, stop=4, num=n_variables[0]).reshape(-1,1) #para que sean n_variables filas y 1 columna
#     #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
#     #Los datos que conocemos son las amplitudes en x=-4 (-inf) x=4 (inf) que son 0
#     X_train = X_grid[[0,-1]]
#     Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array

#     #Analytical solutions definitions
#     #Ground state
#     APsi0 = np.exp(-X_grid**2/2)
#     integral =0
#     for i in range(len(APsi0)):
#         integral = integral + (APsi0[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi0)-1))

#     B = (1/integral)**0.5
#     APsi0 = B * APsi0

#     #1st excitated state
#     APsi1 = np.exp(-X_grid**2/2)*2*X_grid
#     integral =0
#     for i in range(len(APsi1)):
#         integral = integral + (APsi1[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi1)-1))

#     B = (1/integral)**0.5
#     APsi1 = -1 * B * APsi1 #signo menos para correspondencia con solución algoritmo (simetrización)

#     #2nd excitated state
#     APsi2 = np.exp(-X_grid**2/2)*(4*X_grid**2-2)
#     integral =0
#     for i in range(len(APsi2)):
#         integral = integral + (APsi2[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi2)-1))

#     B = (1/integral)**0.5
#     APsi2 = B * APsi2

#     #3rd excitated state
#     APsi3 = np.exp(-X_grid**2/2)*(8*X_grid**3-12*X_grid)
#     integral =0
#     for i in range(len(APsi3)):
#         integral = integral + (APsi3[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi3)-1))

#     B = (1/integral)**0.5
#     APsi3 = -1 * B * APsi3 #signo menos para correspondencia con solución algoritmo (simetrización)

#     APsis = {}
#     APsis['APsi0'] = APsi0[:, 0]
#     APsis['APsi1'] = APsi1[:, 0]
#     APsis['APsi2'] = APsi2[:, 0]
#     APsis['APsi3'] = APsi3[:, 0]

#     ### Ahora el código con EGPA que calcula los 4 primeros estados excitados del problema oscilador armónico
#     params = {'x':X_grid}
#     #Ground state
#     print(f'F should be approx -0.082')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
#     egparesultQHOground = Gaussian.minimize(oneDHOExpE, params, exp_from = 3, output_runtime=False)
#     #Se guardan resultados de algoritmo para siguiente estado excitado
#     Psi_ground=egparesultQHOground.data[-1, 0, :-1]
#     cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
#     E0 = np.sqrt(-np.log(-egparesultQHOground.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así 
#     params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground}
#     print(f'\n  |Psi0> done\n')

#     #1er estado excitado
#     print(f'F should be approx -4.54e-5')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
#     egparesultQHO1 = Gaussian.minimize(oneDHOExpE1, params, exp_from = 5, output_runtime=False)
#     Psi1=egparesultQHO1.data[-1, 0, :-1]
#     cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
#     E1 = np.sqrt(-np.log(-egparesultQHO1.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
#     params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1}
#     print(f'\n  |Psi1> done\n')

#     #2o estado excitado
#     print(f'F should be approx -4.54e-5')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
#     egparesultQHO2 = Gaussian.minimize(oneDHOExpE2, params, exp_from = 5, output_runtime=False)
#     Psi2=egparesultQHO2.data[-1, 0, :-1]
#     cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
#     E2 = np.sqrt(-np.log(-egparesultQHO2.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
#     params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2 }
#     print(f'\n  |Psi2> done\n')

#     #3er estado excitado
#     print(f'F should be approx -4.54e-5')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
#     egparesultQHO3 = Gaussian.minimize(oneDHOExpE3, params, exp_from = 5, output_runtime=False)
#     Psi3=egparesultQHO3.data[-1, 0, :-1]
#     cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
#     E3 = np.sqrt(-np.log(-egparesultQHO3.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
#     params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2, 'E3': E3, 'Psi3': Psi3 }
#     print(f'\n  |Psi3> done\n')

#     #Cálculos de métricas error MSE de cada estado con estado teórico
#     MSEs = []
#     for i in range(4):
#         mse = np.mean((params[f'Psi{i}'] - APsis[f'APsi{i}'])**2)
#         MSEs.append(mse)

#     #MSE_Oscilador = np.sum(MSEs)
#     MSE_Oscilador = np.mean(MSEs)
#     print('\nQuantum Harmonic Oscillator DONE\n')

#     ######################################################################################
#     ######################################################################################
#     ############## POZO DE POTENCIAL INFINITO 1D WELL ####################################
#     ######################################################################################
#     print('\nInfinite Well IN PROGRESS\n')
#     L=5
#     X_grid = np.linspace(start=0, stop=L, num=n_variables[0]).reshape(-1,1) #para que sean n_variables filas y 1 columna
#     #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
#     #Los datos que conocemos son las amplitudes en x=0 x=5 que son 0
#     X_train = X_grid[[0,-1]]
#     Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array
#     params = {'L':L}

#     #Ground state n=1
#     print(f'F should be approx -0.9618')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels ,parallelize=True, init_data=(X_train, Y_train))
#     egparesultgroundWell = Gaussian.minimize(oneD_wellExpE, params, exp_from = 2, output_runtime=False)
#     Psi_fund=egparesultgroundWell.data[-1, 0, :-1]
#     E_fund = np.sqrt(-np.log(-egparesultgroundWell.data[-1,0,-1])) # def exponencial de función de coste la energía se determina a partir del fitness así.
#     #En este caso no hay constante multiplicando exponente exponencial
#     params = {'L':L, 'E': E_fund, 'Psi': Psi_fund}
#     print(f'\n  |Psi0> done\n')

#     #1excitated state n=2
#     print(f'F should be approx -0.7042')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
#     egparesultWellexc1 = Gaussian.minimize(oneD_wellExpEexcitated, params, exp_from = 4, output_runtime=False)
#     Psi_1=egparesultWellexc1.data[-1, 0, :-1]
#     E1 = np.sqrt(-np.log(-egparesultWellexc1.data[-1,0,-1]))
#     params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1}
#     print(f'\n  |Psi1> done\n')

#     #2nd excitated state n=3
#     print(f'F should be approx -0.3775')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
#     egparesultWellexc2 = Gaussian.minimize(oneD_wellExpEexcitated3, params, exp_from = 5, output_runtime=False)
#     Psi_2=egparesultWellexc2.data[-1, 0, :-1]
#     E2 = np.sqrt(-np.log(-egparesultWellexc2.data[-1,0,-1]))
#     params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2}
#     print(f'\n  |Psi2> done\n')

#     #3rd excitated state n=4
#     print(f'F should be approx -0.1482')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
#     egparesultWellexc3 = Gaussian.minimize(oneD_wellExpEexcitated4, params, exp_from = 9, output_runtime=False)
#     Psi_3 = egparesultWellexc3.data[-1, 0, :-1]
#     E3 = np.sqrt(-np.log(-egparesultWellexc3.data[-1,0,-1]))
#     params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2, 'E3':E3, 'Psi3': Psi_3}
#     print(f'\n  |Psi3> done\n')

#     #Soluciones teóricas
#     ### Soluciones analíticas
#     # def Psi_n(n):
#     #     return((2/L)**0.5*np.sin(n*np.pi*np.linspace(0, L, n_variables)/L))

#     APsisWell = {}
#     for n in range(4):
#         APsisWell[f'APsi{n}'] = (2/L)**0.5*np.sin((n+1)*np.pi*np.linspace(0, L, n_variables[0])/L)


#     #Cálculos de métricas error MSE de cada estado con estado teórico
#     MSEs = []
#     for i in range(4):
#         mse = np.mean((params[f'Psi{i}'] - APsisWell[f'APsi{i}'])**2)
#         MSEs.append(mse)

#     #MSE_Well = np.sum(MSEs)
#     #Mean is more correct because sum will make the MSE in NLSE problem probably always smaller 
#     MSE_Well = np.mean(MSEs)

#     print('\nInfinite Well DONE\n')

#     ######################################################################################
#     ######################################################################################
#     #################### N L S E Non linear Schroedinger Equation ########################
#     ######################################################################################
#     print('\nNLSE IN PROGRESS\n')
#     L=2
#     #El grid se define en para los tres problemas de optimización con el mismo número de variables
#     X_grid = np.linspace(start=-1, stop=1, num=n_variables[0]+1).reshape(-1,1) #para que sean n_variables filas y 1 columna
#     #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
#     #Los datos que conocemos son las amplitudes en x=0 que son 1
#     #X_train = X_grid[n_variables//2]
#     X_train = np.array((0)).reshape(-1,1)
#     Y_train = np.linspace(start=1, stop=1, num=1).reshape(-1,1) #Para que sea 2D array
#     params = {'L':L}
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=350, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels ,parallelize=True, init_data=(X_train, Y_train), normalize_solutions=False, symmetrize_solutions=False)
#     egparesultNLSE = Gaussian.minimize(NLSE, params, exp_from = 400, output_runtime=False)
#     #MSE calc
#     #MSE_NLSE = sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.data[-1, 0, :-1])**2) / len(X_grid[:,0])
#     #print(MSE_NLSE, 'using sum')
#     MSE_NLSE = np.sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.data[-1, 0, :-1])**2) / len(X_grid[:,0])
#     MSE_NLSE = np.float64(MSE_NLSE)
#     print('\nNLSE DONE\n')
#     ##################################################################################################

#     #Finally the MSE_global is the accumulated MSE between the three problems
#     MSE_global = [MSE_Oscilador, MSE_Well, MSE_NLSE]
#     print(f'MSE_Oscilador: {round(MSE_Oscilador, 5)}\nMSE_Well: {round(MSE_Well, 5)}\nMSE_NLSE: {round(MSE_NLSE, 5)}\n\n')
#     #Contador para saber cuántas veces se emplea cada problema
#     # Find the index of the max MSE
#     max_index = np.argmax(MSE_global)  # Gets the index of the max value
#     if max_index==0:
#         global timesQOsc
#         timesQOsc+=1
#     if max_index==1:
#         global timesWell
#         timesWell+=1
#     if max_index==2:
#         global timesNLSE
#         timesNLSE+=1
#     #rmse_values = np.random.uniform(0.1, 1.0, 3)  # Replace with real RMSE computations
#     return np.max(MSE_global) #max(rmse_values)  # Min-Max approach: minimize worst RMSE

# global call
# call = 0

# # Wrapper function for Bayesian Optimization
# @use_named_args(search_space)
# def objective(**params):
#     # # Ensure categorical values are correctly interpreted as integers. Categorical was failing when GP takes action because cannot handle categorical GPT suggested this
#     # params["size_gen"] = int(params["size_gen"])
#     # params["dead_iter"] = int(params["dead_iter"])
#     # params["init_kernels"] = int(params["init_kernels"])
#     params['alpha'] = max(0.02, min(0.4, params['alpha']))  
#     params['elite_factor'] = max(0.02, min(0.4, params['elite_factor']))  
#     params['std_lower_bound'] = max(0.01, min(0.2, params['std_lower_bound']))
#     global call
#     call +=1
#     print(f'\n##############################################\nBO call number {call} of 50\n##############################################\n\n')
#     return evaluate_algorithm(**params)

# # Run Bayesian Optimization
# result = gp_minimize(
#     func=objective,      # Objective function
#     dimensions=search_space,  # Search space
#     n_calls=50,         # Number of function evaluations
#     random_state=42,     # For reproducibility
#     n_initial_points=12
# )

# BO_params = {}

# # Best parameters found
# best_params = dict(zip(["size_gen", "dead_iter", "alpha", "elite_factor", "std_lower_bound", "init_kernels"], result.x))
# BO_params[f'{n_variables[0]} variables'] = best_params
# print(f'\n##############################################\nBO of EGPA parameters ended\n##############################################\n\n')
# print(f"Best EGPA Parameters for {n_variables[0]}:", best_params)
# print(f'\nContador\n\nMSE de Oscilador fue usado {timesQOsc} veces\nMSE de Pozo Infinito fue usado {timesWell} veces\nMSE de NLSE fue usado {timesNLSE} veces\n')
# print("Best MSE (Min-Max Approach):", result.fun)



##############################################
BO call number 1 of 50
##############################################


Quantum Harmonic Oscillator IN PROGRESS

F should be approx -0.082
	NFEVALS = 35700 F = [-0.08255353]
	X = [ 8.11413137e-10 -1.78742618e-03 -2.39716223e-03 -1.87379412e-03
 -2.90842474e-04  2.33667988e-03  6.14090117e-03  1.14813706e-02
  1.90031411e-02  2.96561879e-02  4.46615033e-02  6.54123808e-02
  9.33113493e-02  1.29553999e-01  1.74886047e-01  2.29369590e-01
  2.92199845e-01  3.61610488e-01  4.34895251e-01  5.08556346e-01
  5.78569783e-01  6.40739624e-01  6.91097503e-01  7.26296899e-01
  7.43952376e-01  7.42880950e-01  7.23217155e-01  6.86389584e-01
  6.34963685e-01  5.72372194e-01  5.02567028e-01  4.29635675e-01
  3.57428363e-01  2.89239892e-01  2.27581758e-01  1.74066412e-01
  1.29408311e-01  9.35286952e-02  6.57358728e-02  4.49437105e-02
  2.98886873e-02  1.93122923e-02  1.20878463e-02  7.28527309e-03
  4.18255750e-03  2.24183973e-03  1.07262653e-03  4.016346

## BO de parámetros del EGPA para todos los problemas unidimensionales y para 3 número de variables diferentes (3 BO independientes) 
Lo ejecuto por separado

In [2]:
# from EGPA import *
# import numpy as np
# import sys
# import os
# # Get the absolute path of the main directory
# main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# # Add the main directory to sys.path
# sys.path.append(main_dir)
# from OptFunctions.oneDQHarmOscillatorfunctions import *
# from OptFunctions.NonLinearSE import *
# from OptFunctions.oneDwellfunctions import *
# #from utils import _parallel_apply_along_axis
# import time
# import warnings
# warnings.filterwarnings("ignore")

# #np.int = int
# from skopt import gp_minimize
# from skopt.space import Integer, Real, Categorical
# from skopt.utils import use_named_args

# n_variables = [50, 425, 800] #Tamaños del grid (número de dimensiones del problema de optimización)

# # Optimización de parámetros de EGPA
# # Define the search space
# #¿¿?¿?¿?¿?¿?¿?¿?¿? Categorical([10, 30, 60, 100], name="dead_iter") does not work becaues GP cannot handle categorical values
# # search_space = [
# #     Integer(50, 1000, name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
# #     Integer(10, 100, name="dead_iter"),   # Discrete values Categorical([10, 30, 60, 100], name="dead_iter") was not working because GP cannot handle categorical!!!!
# #     Real(0.02, 0.4, name="alpha", prior='log-uniform'),                # Continuous values
# #     Real(0.02, 0.4, name="elite_factor", prior='log-uniform'),        # Continuous values
# #     Real(0.01, 0.2, name="std_lower_bound", prior='log-uniform'),     # Continuous values
# #     Integer(2, 10, name="init_kernels")      # Discrete values
# # ]

# search_space = [
#     Categorical([50, 150, 350, 550, 850, 1000], name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
#     Categorical([10, 30, 60, 100], name="dead_iter"),   # Discrete values Categorical([10, 30, 60, 100], name="dead_iter") was not working because GP cannot handle categorical!!!!
#     Real(0.02, 0.4, name="alpha", prior='log-uniform'),                # Continuous values
#     Real(0.02, 0.4, name="elite_factor", prior='log-uniform'),        # Continuous values
#     Real(0.01, 0.2, name="std_lower_bound", prior='log-uniform'),     # Continuous values
#     Integer(2, 10, name="init_kernels")      # Discrete values
# ]

# global timesQOSc
# timesQOsc = 0
# global timesWell
# timesWell = 0
# global timesNLSE
# timesNLSE = 0

# # Simulated function that runs the optimization and computes RMSE
# def evaluate_algorithm(size_gen, dead_iter, alpha, elite_factor, std_lower_bound, init_kernels):
#     """
#     This function should run the actual optimization algorithm using the given parameters.
#     It should return the worst MSE (max MSE across the three optimization problems).
    
#     Replace the `np.random.uniform()` call with the actual function evaluation.
#     """
#     ##########################################################################################
#     ##########################################################################################
#     ########################### OSCILADOR ARMÓNICO ###########################################
#     ##########################################################################################
#     print('Quantum Harmonic Oscillator IN PROGRESS\n')
#     X_grid = np.linspace(start=-4, stop=4, num=v).reshape(-1,1) #para que sean n_variables filas y 1 columna
#     #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
#     #Los datos que conocemos son las amplitudes en x=-4 (-inf) x=4 (inf) que son 0
#     X_train = X_grid[[0,-1]]
#     Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array

#     #Analytical solutions definitions
#     #Ground state
#     APsi0 = np.exp(-X_grid**2/2)
#     integral =0
#     for i in range(len(APsi0)):
#         integral = integral + (APsi0[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi0)-1))

#     B = (1/integral)**0.5
#     APsi0 = B * APsi0

#     #1st excitated state
#     APsi1 = np.exp(-X_grid**2/2)*2*X_grid
#     integral =0
#     for i in range(len(APsi1)):
#         integral = integral + (APsi1[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi1)-1))

#     B = (1/integral)**0.5
#     APsi1 = -1 * B * APsi1 #signo menos para correspondencia con solución algoritmo (simetrización)

#     #2nd excitated state
#     APsi2 = np.exp(-X_grid**2/2)*(4*X_grid**2-2)
#     integral =0
#     for i in range(len(APsi2)):
#         integral = integral + (APsi2[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi2)-1))

#     B = (1/integral)**0.5
#     APsi2 = B * APsi2

#     #3rd excitated state
#     APsi3 = np.exp(-X_grid**2/2)*(8*X_grid**3-12*X_grid)
#     integral =0
#     for i in range(len(APsi3)):
#         integral = integral + (APsi3[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi3)-1))

#     B = (1/integral)**0.5
#     APsi3 = -1 * B * APsi3 #signo menos para correspondencia con solución algoritmo (simetrización)

#     APsis = {}
#     APsis['APsi0'] = APsi0[:, 0]
#     APsis['APsi1'] = APsi1[:, 0]
#     APsis['APsi2'] = APsi2[:, 0]
#     APsis['APsi3'] = APsi3[:, 0]

#     ### Ahora el código con EGPA que calcula los 4 primeros estados excitados del problema oscilador armónico
#     params = {'x':X_grid}
#     #Ground state
#     print(f'F should be approx -0.082')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
#     egparesultQHOground = Gaussian.minimize(oneDHOExpE, params, exp_from = 3, output_runtime=False)
#     #Se guardan resultados de algoritmo para siguiente estado excitado
#     Psi_ground=egparesultQHOground.data[-1, 0, :-1]
#     cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
#     E0 = np.sqrt(-np.log(-egparesultQHOground.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así 
#     params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground}
#     print(f'\n  |Psi0> done\n')

#     #1er estado excitado
#     print(f'F should be approx -4.54e-5')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
#     egparesultQHO1 = Gaussian.minimize(oneDHOExpE1, params, exp_from = 5, output_runtime=False)
#     Psi1=egparesultQHO1.data[-1, 0, :-1]
#     cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
#     E1 = np.sqrt(-np.log(-egparesultQHO1.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
#     params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1}
#     print(f'\n  |Psi1> done\n')

#     #2o estado excitado
#     print(f'F should be approx -4.54e-5')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
#     egparesultQHO2 = Gaussian.minimize(oneDHOExpE2, params, exp_from = 5, output_runtime=False)
#     Psi2=egparesultQHO2.data[-1, 0, :-1]
#     cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
#     E2 = np.sqrt(-np.log(-egparesultQHO2.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
#     params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2 }
#     print(f'\n  |Psi2> done\n')

#     #3er estado excitado
#     print(f'F should be approx -4.54e-5')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
#     egparesultQHO3 = Gaussian.minimize(oneDHOExpE3, params, exp_from = 5, output_runtime=False)
#     Psi3=egparesultQHO3.data[-1, 0, :-1]
#     cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
#     E3 = np.sqrt(-np.log(-egparesultQHO3.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
#     params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2, 'E3': E3, 'Psi3': Psi3 }
#     print(f'\n  |Psi3> done\n')

#     #Cálculos de métricas error MSE de cada estado con estado teórico
#     MSEs = []
#     for i in range(4):
#         mse = np.mean((params[f'Psi{i}'] - APsis[f'APsi{i}'])**2)
#         MSEs.append(mse)

#     #MSE_Oscilador = np.sum(MSEs)
#     MSE_Oscilador = np.mean(MSEs)
#     print('\nQuantum Harmonic Oscillator DONE\n')

#     ######################################################################################
#     ######################################################################################
#     ############## POZO DE POTENCIAL INFINITO 1D WELL ####################################
#     ######################################################################################
#     print('\nInfinite Well IN PROGRESS\n')
#     L=5
#     X_grid = np.linspace(start=0, stop=L, num=v).reshape(-1,1) #para que sean n_variables filas y 1 columna
#     #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
#     #Los datos que conocemos son las amplitudes en x=0 x=5 que son 0
#     X_train = X_grid[[0,-1]]
#     Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array
#     params = {'L':L}

#     #Ground state n=1
#     print(f'F should be approx -0.9618')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels ,parallelize=True, init_data=(X_train, Y_train))
#     egparesultgroundWell = Gaussian.minimize(oneD_wellExpE, params, exp_from = 2, output_runtime=False)
#     Psi_fund=egparesultgroundWell.data[-1, 0, :-1]
#     E_fund = np.sqrt(-np.log(-egparesultgroundWell.data[-1,0,-1])) # def exponencial de función de coste la energía se determina a partir del fitness así.
#     #En este caso no hay constante multiplicando exponente exponencial
#     params = {'L':L, 'E': E_fund, 'Psi': Psi_fund}
#     print(f'\n  |Psi0> done\n')

#     #1excitated state n=2
#     print(f'F should be approx -0.7042')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
#     egparesultWellexc1 = Gaussian.minimize(oneD_wellExpEexcitated, params, exp_from = 4, output_runtime=False)
#     Psi_1=egparesultWellexc1.data[-1, 0, :-1]
#     E1 = np.sqrt(-np.log(-egparesultWellexc1.data[-1,0,-1]))
#     params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1}
#     print(f'\n  |Psi1> done\n')

#     #2nd excitated state n=3
#     print(f'F should be approx -0.3775')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
#     egparesultWellexc2 = Gaussian.minimize(oneD_wellExpEexcitated3, params, exp_from = 5, output_runtime=False)
#     Psi_2=egparesultWellexc2.data[-1, 0, :-1]
#     E2 = np.sqrt(-np.log(-egparesultWellexc2.data[-1,0,-1]))
#     params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2}
#     print(f'\n  |Psi2> done\n')

#     #3rd excitated state n=4
#     print(f'F should be approx -0.1482')
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
#     egparesultWellexc3 = Gaussian.minimize(oneD_wellExpEexcitated4, params, exp_from = 9, output_runtime=False)
#     Psi_3 = egparesultWellexc3.data[-1, 0, :-1]
#     E3 = np.sqrt(-np.log(-egparesultWellexc3.data[-1,0,-1]))
#     params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2, 'E3':E3, 'Psi3': Psi_3}
#     print(f'\n  |Psi3> done\n')

#     #Soluciones teóricas
#     ### Soluciones analíticas
#     # def Psi_n(n):
#     #     return((2/L)**0.5*np.sin(n*np.pi*np.linspace(0, L, n_variables)/L))

#     APsisWell = {}
#     for n in range(4):
#         APsisWell[f'APsi{n}'] = (2/L)**0.5*np.sin((n+1)*np.pi*np.linspace(0, L, v)/L)


#     #Cálculos de métricas error MSE de cada estado con estado teórico
#     MSEs = []
#     for i in range(4):
#         mse = np.mean((params[f'Psi{i}'] - APsisWell[f'APsi{i}'])**2)
#         MSEs.append(mse)

#     #MSE_Well = np.sum(MSEs)
#     #Mean is more correct because sum will make the MSE in NLSE problem probably always smaller 
#     MSE_Well = np.mean(MSEs)

#     print('\nInfinite Well DONE\n')

#     ######################################################################################
#     ######################################################################################
#     #################### N L S E Non linear Schroedinger Equation ########################
#     ######################################################################################
#     print('\nNLSE IN PROGRESS\n')
#     L=2
#     #El grid se define en para los tres problemas de optimización con el mismo número de variables
#     X_grid = np.linspace(start=-1, stop=1, num=v+1).reshape(-1,1) #para que sean n_variables filas y 1 columna
#     #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
#     #Los datos que conocemos son las amplitudes en x=0 que son 1
#     #X_train = X_grid[n_variables//2]
#     X_train = np.array((0)).reshape(-1,1)
#     Y_train = np.linspace(start=1, stop=1, num=1).reshape(-1,1) #Para que sea 2D array
#     params = {'L':L}
#     Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=350, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels ,parallelize=True, init_data=(X_train, Y_train), normalize_solutions=False, symmetrize_solutions=False)
#     egparesultNLSE = Gaussian.minimize(NLSE, params, exp_from = 400, output_runtime=False)
#     #MSE calc
#     #MSE_NLSE = sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.data[-1, 0, :-1])**2) / len(X_grid[:,0])
#     #print(MSE_NLSE, 'using sum')
#     MSE_NLSE = np.sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.data[-1, 0, :-1])**2) / len(X_grid[:,0])
#     MSE_NLSE = np.float64(MSE_NLSE)
#     print('\nNLSE DONE\n')
#     ##################################################################################################

#     #Finally the MSE_global is the accumulated MSE between the three problems
#     MSE_global = [MSE_Oscilador, MSE_Well, MSE_NLSE]
#     print(f'MSE_Oscilador: {round(MSE_Oscilador, 5)}\nMSE_Well: {round(MSE_Well, 5)}\nMSE_NLSE: {round(MSE_NLSE, 5)}\n\n')
#     #Contador para saber cuántas veces se emplea cada problema
#     # Find the index of the max MSE
#     max_index = np.argmax(MSE_global)  # Gets the index of the max value
#     if max_index==0:
#         global timesQOsc
#         timesQOsc+=1
#     if max_index==1:
#         global timesWell
#         timesWell+=1
#     if max_index==2:
#         global timesNLSE
#         timesNLSE+=1
#     #rmse_values = np.random.uniform(0.1, 1.0, 3)  # Replace with real RMSE computations
#     return np.max(MSE_global) #max(rmse_values)  # Min-Max approach: minimize worst RMSE

# global call
# call = 0

# # Wrapper function for Bayesian Optimization
# @use_named_args(search_space)
# def objective(**params):
#     # # Ensure categorical values are correctly interpreted as integers. Categorical was failing when GP takes action because cannot handle categorical GPT suggested this
#     # params["size_gen"] = int(params["size_gen"])
#     # params["dead_iter"] = int(params["dead_iter"])
#     # params["init_kernels"] = int(params["init_kernels"])
#     params['alpha'] = max(0.02, min(0.4, params['alpha']))  
#     params['elite_factor'] = max(0.02, min(0.4, params['elite_factor']))  
#     params['std_lower_bound'] = max(0.01, min(0.2, params['std_lower_bound']))
#     global call
#     call +=1
#     print(f'\n##############################################\nBO call number {call} of 150\n##############################################\n\n')
#     return evaluate_algorithm(**params)

# BO_params = {}

# for v in n_variables:
#     # Run Bayesian Optimization
#     print(f'\n##############################################\nBO for {v} variables\n##############################################\n\n')
#     result = gp_minimize(
#         func=objective,      # Objective function
#         dimensions=search_space,  # Search space
#         n_calls=50,         # Number of function evaluations
#         random_state=42,     # For reproducibility
#         n_initial_points=12
#     )

#     # Best parameters found
#     best_params = dict(zip(["size_gen", "dead_iter", "alpha", "elite_factor", "std_lower_bound", "init_kernels"], result.x))
#     BO_params[f'{v} variables'] = best_params
#     BO_params[f'{v} variables MSE problem instances counts'] = f'MSE de Oscilador fue usado {timesQOsc} veces. MSE de Pozo Infinito fue usado {timesWell} veces. MSE de NLSE fue usado {timesNLSE} veces'
#     print(f'\n##############################################\nBO of EGPA parameters ended\n##############################################\n\n')
#     print(f"Best EGPA Parameters for {v}:", best_params)
#     print(f'\nContador\n\nMSE de Oscilador fue usado {timesQOsc} veces\nMSE de Pozo Infinito fue usado {timesWell} veces\nMSE de NLSE fue usado {timesNLSE} veces\n')
#     print("Best MSE (Min-Max Approach):", result.fun)


# for keys in BO_params.keys():
#     print(f'Problema {keys}: {BO_params[keys]}\n')


In [1]:
from EGPA import *
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Add the main directory to sys.path
sys.path.append(main_dir)
from OptFunctions.oneDQHarmOscillatorfunctions import *
from OptFunctions.NonLinearSE import *
from OptFunctions.oneDwellfunctions import *
#from utils import _parallel_apply_along_axis
import time
import warnings
warnings.filterwarnings("ignore")

#np.int = int
from skopt import gp_minimize
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args

n_variables = [50, 425, 800] #Tamaños del grid (número de dimensiones del problema de optimización)

# Optimización de parámetros de EGPA
# Define the search space
#¿¿?¿?¿?¿?¿?¿?¿?¿? Categorical([10, 30, 60, 100], name="dead_iter") does not work becaues GP cannot handle categorical values
# search_space = [
#     Integer(50, 1000, name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
#     Integer(10, 100, name="dead_iter"),   # Discrete values Categorical([10, 30, 60, 100], name="dead_iter") was not working because GP cannot handle categorical!!!!
#     Real(0.02, 0.4, name="alpha", prior='log-uniform'),                # Continuous values
#     Real(0.02, 0.4, name="elite_factor", prior='log-uniform'),        # Continuous values
#     Real(0.01, 0.2, name="std_lower_bound", prior='log-uniform'),     # Continuous values
#     Integer(2, 10, name="init_kernels")      # Discrete values
# ]

search_space = [
    Categorical([50, 150, 350, 550, 850, 1000], name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
    Categorical([10, 30, 60, 100], name="dead_iter"),   # Discrete values Categorical([10, 30, 60, 100], name="dead_iter") was not working because GP cannot handle categorical!!!!
    Real(0.02, 0.4, name="alpha", prior='log-uniform'),                # Continuous values
    Real(0.02, 0.4, name="elite_factor", prior='log-uniform'),        # Continuous values
    Real(0.01, 0.2, name="std_lower_bound", prior='log-uniform'),     # Continuous values
    Integer(2, 10, name="init_kernels")      # Discrete values
]

global timesQOSc
timesQOsc = 0
global timesWell
timesWell = 0
global timesNLSE
timesNLSE = 0

# Simulated function that runs the optimization and computes RMSE
def evaluate_algorithm(size_gen, dead_iter, alpha, elite_factor, std_lower_bound, init_kernels):
    """
    This function should run the actual optimization algorithm using the given parameters.
    It should return the worst MSE (max MSE across the three optimization problems).
    
    Replace the `np.random.uniform()` call with the actual function evaluation.
    """
    ##########################################################################################
    ##########################################################################################
    ########################### OSCILADOR ARMÓNICO ###########################################
    ##########################################################################################
    print('Quantum Harmonic Oscillator IN PROGRESS\n')
    X_grid = np.linspace(start=-4, stop=4, num=n_variables[0]).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=-4 (-inf) x=4 (inf) que son 0
    X_train = X_grid[[0,-1]]
    Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array

    #Analytical solutions definitions
    #Ground state
    APsi0 = np.exp(-X_grid**2/2)
    integral =0
    for i in range(len(APsi0)):
        integral = integral + (APsi0[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi0)-1))

    B = (1/integral)**0.5
    APsi0 = B * APsi0

    #1st excitated state
    APsi1 = np.exp(-X_grid**2/2)*2*X_grid
    integral =0
    for i in range(len(APsi1)):
        integral = integral + (APsi1[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi1)-1))

    B = (1/integral)**0.5
    APsi1 = -1 * B * APsi1 #signo menos para correspondencia con solución algoritmo (simetrización)

    #2nd excitated state
    APsi2 = np.exp(-X_grid**2/2)*(4*X_grid**2-2)
    integral =0
    for i in range(len(APsi2)):
        integral = integral + (APsi2[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi2)-1))

    B = (1/integral)**0.5
    APsi2 = B * APsi2

    #3rd excitated state
    APsi3 = np.exp(-X_grid**2/2)*(8*X_grid**3-12*X_grid)
    integral =0
    for i in range(len(APsi3)):
        integral = integral + (APsi3[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi3)-1))

    B = (1/integral)**0.5
    APsi3 = -1 * B * APsi3 #signo menos para correspondencia con solución algoritmo (simetrización)

    APsis = {}
    APsis['APsi0'] = APsi0[:, 0]
    APsis['APsi1'] = APsi1[:, 0]
    APsis['APsi2'] = APsi2[:, 0]
    APsis['APsi3'] = APsi3[:, 0]

    ### Ahora el código con EGPA que calcula los 4 primeros estados excitados del problema oscilador armónico
    params = {'x':X_grid}
    #Ground state
    print(f'F should be approx -0.082')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHOground = Gaussian.minimize(oneDHOExpE, params, exp_from = 3, output_runtime=False)
    #Se guardan resultados de algoritmo para siguiente estado excitado
    Psi_ground=egparesultQHOground.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E0 = np.sqrt(-np.log(-egparesultQHOground.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así 
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground}
    print(f'\n  |Psi0> done\n')

    #1er estado excitado
    print(f'F should be approx -4.54e-5')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHO1 = Gaussian.minimize(oneDHOExpE1, params, exp_from = 5, output_runtime=False)
    Psi1=egparesultQHO1.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E1 = np.sqrt(-np.log(-egparesultQHO1.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1}
    print(f'\n  |Psi1> done\n')

    #2o estado excitado
    print(f'F should be approx -4.54e-5')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHO2 = Gaussian.minimize(oneDHOExpE2, params, exp_from = 5, output_runtime=False)
    Psi2=egparesultQHO2.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E2 = np.sqrt(-np.log(-egparesultQHO2.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2 }
    print(f'\n  |Psi2> done\n')

    #3er estado excitado
    print(f'F should be approx -4.54e-5')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHO3 = Gaussian.minimize(oneDHOExpE3, params, exp_from = 5, output_runtime=False)
    Psi3=egparesultQHO3.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E3 = np.sqrt(-np.log(-egparesultQHO3.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2, 'E3': E3, 'Psi3': Psi3 }
    print(f'\n  |Psi3> done\n')

    #Cálculos de métricas error MSE de cada estado con estado teórico
    MSEs = []
    for i in range(4):
        mse = np.mean((params[f'Psi{i}'] - APsis[f'APsi{i}'])**2)
        MSEs.append(mse)

    #MSE_Oscilador = np.sum(MSEs)
    MSE_Oscilador = np.mean(MSEs)
    print('\nQuantum Harmonic Oscillator DONE\n')

    ######################################################################################
    ######################################################################################
    ############## POZO DE POTENCIAL INFINITO 1D WELL ####################################
    ######################################################################################
    print('\nInfinite Well IN PROGRESS\n')
    L=5
    X_grid = np.linspace(start=0, stop=L, num=n_variables[0]).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=0 x=5 que son 0
    X_train = X_grid[[0,-1]]
    Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array
    params = {'L':L}

    #Ground state n=1
    print(f'F should be approx -0.9618')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels ,parallelize=True, init_data=(X_train, Y_train))
    egparesultgroundWell = Gaussian.minimize(oneD_wellExpE, params, exp_from = 2, output_runtime=False)
    Psi_fund=egparesultgroundWell.data[-1, 0, :-1]
    E_fund = np.sqrt(-np.log(-egparesultgroundWell.data[-1,0,-1])) # def exponencial de función de coste la energía se determina a partir del fitness así.
    #En este caso no hay constante multiplicando exponente exponencial
    params = {'L':L, 'E': E_fund, 'Psi': Psi_fund}
    print(f'\n  |Psi0> done\n')

    #1excitated state n=2
    print(f'F should be approx -0.7042')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
    egparesultWellexc1 = Gaussian.minimize(oneD_wellExpEexcitated, params, exp_from = 4, output_runtime=False)
    Psi_1=egparesultWellexc1.data[-1, 0, :-1]
    E1 = np.sqrt(-np.log(-egparesultWellexc1.data[-1,0,-1]))
    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1}
    print(f'\n  |Psi1> done\n')

    #2nd excitated state n=3
    print(f'F should be approx -0.3775')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
    egparesultWellexc2 = Gaussian.minimize(oneD_wellExpEexcitated3, params, exp_from = 5, output_runtime=False)
    Psi_2=egparesultWellexc2.data[-1, 0, :-1]
    E2 = np.sqrt(-np.log(-egparesultWellexc2.data[-1,0,-1]))
    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2}
    print(f'\n  |Psi2> done\n')

    #3rd excitated state n=4
    print(f'F should be approx -0.1482')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=300, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
    egparesultWellexc3 = Gaussian.minimize(oneD_wellExpEexcitated4, params, exp_from = 9, output_runtime=False)
    Psi_3 = egparesultWellexc3.data[-1, 0, :-1]
    E3 = np.sqrt(-np.log(-egparesultWellexc3.data[-1,0,-1]))
    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2, 'E3':E3, 'Psi3': Psi_3}
    print(f'\n  |Psi3> done\n')

    #Soluciones teóricas
    ### Soluciones analíticas
    # def Psi_n(n):
    #     return((2/L)**0.5*np.sin(n*np.pi*np.linspace(0, L, n_variables)/L))

    APsisWell = {}
    for n in range(4):
        APsisWell[f'APsi{n}'] = (2/L)**0.5*np.sin((n+1)*np.pi*np.linspace(0, L, n_variables[0])/L)


    #Cálculos de métricas error MSE de cada estado con estado teórico
    MSEs = []
    for i in range(4):
        mse = np.mean((params[f'Psi{i}'] - APsisWell[f'APsi{i}'])**2)
        MSEs.append(mse)

    #MSE_Well = np.sum(MSEs)
    #Mean is more correct because sum will make the MSE in NLSE problem probably always smaller 
    MSE_Well = np.mean(MSEs)

    print('\nInfinite Well DONE\n')

    ######################################################################################
    ######################################################################################
    #################### N L S E Non linear Schroedinger Equation ########################
    ######################################################################################
    print('\nNLSE IN PROGRESS\n')
    L=2
    #El grid se define en para los tres problemas de optimización con el mismo número de variables
    X_grid = np.linspace(start=-1, stop=1, num=n_variables[0]+1).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=0 que son 1
    #X_train = X_grid[n_variables//2]
    X_train = np.array((0)).reshape(-1,1)
    Y_train = np.linspace(start=1, stop=1, num=1).reshape(-1,1) #Para que sea 2D array
    params = {'L':L}
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=350, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels ,parallelize=True, init_data=(X_train, Y_train), normalize_solutions=False, symmetrize_solutions=False)
    egparesultNLSE = Gaussian.minimize(NLSE, params, exp_from = 400, output_runtime=False)
    #MSE calc
    #MSE_NLSE = sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.data[-1, 0, :-1])**2) / len(X_grid[:,0])
    #print(MSE_NLSE, 'using sum')
    MSE_NLSE = np.sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.data[-1, 0, :-1])**2) / len(X_grid[:,0])
    MSE_NLSE = np.float64(MSE_NLSE)
    print('\nNLSE DONE\n')
    ##################################################################################################

    #Finally the MSE_global is the accumulated MSE between the three problems
    MSE_global = [MSE_Oscilador, MSE_Well, MSE_NLSE]
    print(f'MSE_Oscilador: {round(MSE_Oscilador, 5)}\nMSE_Well: {round(MSE_Well, 5)}\nMSE_NLSE: {round(MSE_NLSE, 5)}\n\n')
    #Contador para saber cuántas veces se emplea cada problema
    # Find the index of the max MSE
    max_index = np.argmax(MSE_global)  # Gets the index of the max value
    if max_index==0:
        global timesQOsc
        timesQOsc+=1
    if max_index==1:
        global timesWell
        timesWell+=1
    if max_index==2:
        global timesNLSE
        timesNLSE+=1
    #rmse_values = np.random.uniform(0.1, 1.0, 3)  # Replace with real RMSE computations
    return np.max(MSE_global) #max(rmse_values)  # Min-Max approach: minimize worst RMSE

global call
call = 0

# Wrapper function for Bayesian Optimization
@use_named_args(search_space)
def objective(**params):
    # # Ensure categorical values are correctly interpreted as integers. Categorical was failing when GP takes action because cannot handle categorical GPT suggested this
    # params["size_gen"] = int(params["size_gen"])
    # params["dead_iter"] = int(params["dead_iter"])
    # params["init_kernels"] = int(params["init_kernels"])
    params['alpha'] = max(0.02, min(0.4, params['alpha']))  
    params['elite_factor'] = max(0.02, min(0.4, params['elite_factor']))  
    params['std_lower_bound'] = max(0.01, min(0.2, params['std_lower_bound']))
    global call
    call +=1
    print(f'\n##############################################\nBO call number {call} of 50\n##############################################\n\n')
    return evaluate_algorithm(**params)

BO_params50 = {}

# Run Bayesian Optimization
print(f'\n##############################################\nBO for {n_variables[0]} variables\n##############################################\n\n')
result = gp_minimize(
    func=objective,      # Objective function
    dimensions=search_space,  # Search space
    n_calls=50,         # Number of function evaluations
    random_state=42,     # For reproducibility
    n_initial_points=12#, n_jobs=-1 #Number of cores to run in parallel while running the lbfgs optimizations over the acquisition function. Valid only when acq_optimizer is set to "lbfgs". Defaults to 1 core. If n_jobs=-1, then number of jobs is set to number of cores.
)

# Best parameters found
best_params = dict(zip(["size_gen", "dead_iter", "alpha", "elite_factor", "std_lower_bound", "init_kernels"], result.x))
BO_params50['params'] = best_params
BO_params50['problem instances counts'] = f'MSE de Oscilador fue usado {timesQOsc} veces. MSE de Pozo Infinito fue usado {timesWell} veces. MSE de NLSE fue usado {timesNLSE} veces'
print(f'\n##############################################\nBO of EGPA parameters ended\n##############################################\n\n')
print(f"Best EGPA Parameters for {n_variables[0]}:", best_params)
print(f'\nContador\n\nMSE de Oscilador fue usado {timesQOsc} veces\nMSE de Pozo Infinito fue usado {timesWell} veces\nMSE de NLSE fue usado {timesNLSE} veces\n')
print("Best MSE (Min-Max Approach):", result.fun)

import pickle
ruta = 'BO_params/50EGPA'
with open(f'{ruta}.pkl', "wb") as file:
    pickle.dump(BO_params50, file)


for keys in BO_params50.keys():
    print(f'Problema {keys}: {BO_params50[keys]}\n')



##############################################
BO for 50 variables
##############################################



##############################################
BO call number 1 of 50
##############################################


Quantum Harmonic Oscillator IN PROGRESS

F should be approx -0.082
	NFEVALS = 42500 F = [-0.08264816]
	X = [ 3.72298000e-11 -4.35682786e-04 -4.29077011e-04  7.12820728e-05
  1.22232404e-03  3.27306106e-03  6.57534785e-03  1.16179294e-02
  1.90711154e-02  2.98208169e-02  4.49676778e-02  6.57696383e-02
  9.35172990e-02  1.29346952e-01  1.74012775e-01  2.27653345e-01
  2.89595353e-01  3.58235812e-01  4.31033414e-01  5.04624948e-01
  5.75061217e-01  6.38140216e-01  6.89798623e-01  7.26515340e-01
  7.45676557e-01  7.45857214e-01  7.26982452e-01  6.90346744e-01
  6.38486116e-01  5.74916855e-01  5.03772712e-01  4.29385932e-01
  3.55867079e-01  2.86737190e-01  2.24658740e-01  1.71294240e-01
  1.27301371e-01  9.24506087e-02  6.58340127e-02  4.61222097e-02
  3.18

In [1]:
from EGPA import *
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Add the main directory to sys.path
sys.path.append(main_dir)
from OptFunctions.oneDQHarmOscillatorfunctions import *
from OptFunctions.NonLinearSE import *
from OptFunctions.oneDwellfunctions import *
#from utils import _parallel_apply_along_axis
import time
import warnings
warnings.filterwarnings("ignore")

#np.int = int
from skopt import gp_minimize
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args

n_variables = [50, 425, 800] #Tamaños del grid (número de dimensiones del problema de optimización)

# Optimización de parámetros de EGPA
# Define the search space
#¿¿?¿?¿?¿?¿?¿?¿?¿? Categorical([10, 30, 60, 100], name="dead_iter") does not work becaues GP cannot handle categorical values
# search_space = [
#     Integer(50, 1000, name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
#     Integer(10, 100, name="dead_iter"),   # Discrete values Categorical([10, 30, 60, 100], name="dead_iter") was not working because GP cannot handle categorical!!!!
#     Real(0.02, 0.4, name="alpha", prior='log-uniform'),                # Continuous values
#     Real(0.02, 0.4, name="elite_factor", prior='log-uniform'),        # Continuous values
#     Real(0.01, 0.2, name="std_lower_bound", prior='log-uniform'),     # Continuous values
#     Integer(2, 10, name="init_kernels")      # Discrete values
# ]

search_space = [
    Categorical([50, 150, 350, 550, 850, 1000], name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
    Categorical([10, 30, 60, 100], name="dead_iter"),   # Discrete values Categorical([10, 30, 60, 100], name="dead_iter") was not working because GP cannot handle categorical!!!!
    Real(0.02, 0.4, name="alpha", prior='log-uniform'),                # Continuous values
    Real(0.02, 0.4, name="elite_factor", prior='log-uniform'),        # Continuous values
    Real(0.01, 0.2, name="std_lower_bound", prior='log-uniform'),     # Continuous values
    Integer(2, 10, name="init_kernels")      # Discrete values
]

global timesQOSc
timesQOsc = 0
global timesWell
timesWell = 0
global timesNLSE
timesNLSE = 0

# Simulated function that runs the optimization and computes RMSE
def evaluate_algorithm(size_gen, dead_iter, alpha, elite_factor, std_lower_bound, init_kernels):
    """
    This function should run the actual optimization algorithm using the given parameters.
    It should return the worst MSE (max MSE across the three optimization problems).
    
    Replace the `np.random.uniform()` call with the actual function evaluation.
    """
    ##########################################################################################
    ##########################################################################################
    ########################### OSCILADOR ARMÓNICO ###########################################
    ##########################################################################################
    print('Quantum Harmonic Oscillator IN PROGRESS\n')
    X_grid = np.linspace(start=-4, stop=4, num=n_variables[1]).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=-4 (-inf) x=4 (inf) que son 0
    X_train = X_grid[[0,-1]]
    Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array

    #Analytical solutions definitions
    #Ground state
    APsi0 = np.exp(-X_grid**2/2)
    integral =0
    for i in range(len(APsi0)):
        integral = integral + (APsi0[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi0)-1))

    B = (1/integral)**0.5
    APsi0 = B * APsi0

    #1st excitated state
    APsi1 = np.exp(-X_grid**2/2)*2*X_grid
    integral =0
    for i in range(len(APsi1)):
        integral = integral + (APsi1[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi1)-1))

    B = (1/integral)**0.5
    APsi1 = -1 * B * APsi1 #signo menos para correspondencia con solución algoritmo (simetrización)

    #2nd excitated state
    APsi2 = np.exp(-X_grid**2/2)*(4*X_grid**2-2)
    integral =0
    for i in range(len(APsi2)):
        integral = integral + (APsi2[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi2)-1))

    B = (1/integral)**0.5
    APsi2 = B * APsi2

    #3rd excitated state
    APsi3 = np.exp(-X_grid**2/2)*(8*X_grid**3-12*X_grid)
    integral =0
    for i in range(len(APsi3)):
        integral = integral + (APsi3[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi3)-1))

    B = (1/integral)**0.5
    APsi3 = -1 * B * APsi3 #signo menos para correspondencia con solución algoritmo (simetrización)

    APsis = {}
    APsis['APsi0'] = APsi0[:, 0]
    APsis['APsi1'] = APsi1[:, 0]
    APsis['APsi2'] = APsi2[:, 0]
    APsis['APsi3'] = APsi3[:, 0]

    ### Ahora el código con EGPA que calcula los 4 primeros estados excitados del problema oscilador armónico
    params = {'x':X_grid}
    #Ground state
    print(f'F should be approx -0.082')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHOground = Gaussian.minimize(oneDHOExpE, params, exp_from = 3, output_runtime=False)
    #Se guardan resultados de algoritmo para siguiente estado excitado
    Psi_ground=egparesultQHOground.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E0 = np.sqrt(-np.log(-egparesultQHOground.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así 
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground}
    print(f'\n  |Psi0> done\n')

    #1er estado excitado
    print(f'F should be approx -4.54e-5')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHO1 = Gaussian.minimize(oneDHOExpE1, params, exp_from = 5, output_runtime=False)
    Psi1=egparesultQHO1.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E1 = np.sqrt(-np.log(-egparesultQHO1.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1}
    print(f'\n  |Psi1> done\n')

    #2o estado excitado
    print(f'F should be approx -4.54e-5')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHO2 = Gaussian.minimize(oneDHOExpE2, params, exp_from = 5, output_runtime=False)
    Psi2=egparesultQHO2.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E2 = np.sqrt(-np.log(-egparesultQHO2.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2 }
    print(f'\n  |Psi2> done\n')

    #3er estado excitado
    print(f'F should be approx -4.54e-5')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHO3 = Gaussian.minimize(oneDHOExpE3, params, exp_from = 5, output_runtime=False)
    Psi3=egparesultQHO3.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E3 = np.sqrt(-np.log(-egparesultQHO3.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2, 'E3': E3, 'Psi3': Psi3 }
    print(f'\n  |Psi3> done\n')

    #Cálculos de métricas error MSE de cada estado con estado teórico
    MSEs = []
    for i in range(4):
        mse = np.mean((params[f'Psi{i}'] - APsis[f'APsi{i}'])**2)
        MSEs.append(mse)

    #MSE_Oscilador = np.sum(MSEs)
    MSE_Oscilador = np.mean(MSEs)
    print('\nQuantum Harmonic Oscillator DONE\n')

    ######################################################################################
    ######################################################################################
    ############## POZO DE POTENCIAL INFINITO 1D WELL ####################################
    ######################################################################################
    print('\nInfinite Well IN PROGRESS\n')
    L=5
    X_grid = np.linspace(start=0, stop=L, num=n_variables[1]).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=0 x=5 que son 0
    X_train = X_grid[[0,-1]]
    Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array
    params = {'L':L}

    #Ground state n=1
    print(f'F should be approx -0.9618')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels ,parallelize=True, init_data=(X_train, Y_train))
    egparesultgroundWell = Gaussian.minimize(oneD_wellExpE, params, exp_from = 2, output_runtime=False)
    Psi_fund=egparesultgroundWell.data[-1, 0, :-1]
    E_fund = np.sqrt(-np.log(-egparesultgroundWell.data[-1,0,-1])) # def exponencial de función de coste la energía se determina a partir del fitness así.
    #En este caso no hay constante multiplicando exponente exponencial
    params = {'L':L, 'E': E_fund, 'Psi': Psi_fund}
    print(f'\n  |Psi0> done\n')

    #1excitated state n=2
    print(f'F should be approx -0.7042')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
    egparesultWellexc1 = Gaussian.minimize(oneD_wellExpEexcitated, params, exp_from = 4, output_runtime=False)
    Psi_1=egparesultWellexc1.data[-1, 0, :-1]
    E1 = np.sqrt(-np.log(-egparesultWellexc1.data[-1,0,-1]))
    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1}
    print(f'\n  |Psi1> done\n')

    #2nd excitated state n=3
    print(f'F should be approx -0.3775')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
    egparesultWellexc2 = Gaussian.minimize(oneD_wellExpEexcitated3, params, exp_from = 5, output_runtime=False)
    Psi_2=egparesultWellexc2.data[-1, 0, :-1]
    E2 = np.sqrt(-np.log(-egparesultWellexc2.data[-1,0,-1]))
    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2}
    print(f'\n  |Psi2> done\n')

    #3rd excitated state n=4
    print(f'F should be approx -0.1482')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
    egparesultWellexc3 = Gaussian.minimize(oneD_wellExpEexcitated4, params, exp_from = 9, output_runtime=False)
    Psi_3 = egparesultWellexc3.data[-1, 0, :-1]
    E3 = np.sqrt(-np.log(-egparesultWellexc3.data[-1,0,-1]))
    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2, 'E3':E3, 'Psi3': Psi_3}
    print(f'\n  |Psi3> done\n')

    #Soluciones teóricas
    ### Soluciones analíticas
    # def Psi_n(n):
    #     return((2/L)**0.5*np.sin(n*np.pi*np.linspace(0, L, n_variables)/L))

    APsisWell = {}
    for n in range(4):
        APsisWell[f'APsi{n}'] = (2/L)**0.5*np.sin((n+1)*np.pi*np.linspace(0, L, n_variables[1])/L)


    #Cálculos de métricas error MSE de cada estado con estado teórico
    MSEs = []
    for i in range(4):
        mse = np.mean((params[f'Psi{i}'] - APsisWell[f'APsi{i}'])**2)
        MSEs.append(mse)

    #MSE_Well = np.sum(MSEs)
    #Mean is more correct because sum will make the MSE in NLSE problem probably always smaller 
    MSE_Well = np.mean(MSEs)

    print('\nInfinite Well DONE\n')

    ######################################################################################
    ######################################################################################
    #################### N L S E Non linear Schroedinger Equation ########################
    ######################################################################################
    print('\nNLSE IN PROGRESS\n')
    L=2
    #El grid se define en para los tres problemas de optimización con el mismo número de variables
    X_grid = np.linspace(start=-1, stop=1, num=n_variables[1]+1).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=0 que son 1
    #X_train = X_grid[n_variables//2]
    X_train = np.array((0)).reshape(-1,1)
    Y_train = np.linspace(start=1, stop=1, num=1).reshape(-1,1) #Para que sea 2D array
    params = {'L':L}
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels ,parallelize=True, init_data=(X_train, Y_train), normalize_solutions=False, symmetrize_solutions=False)
    egparesultNLSE = Gaussian.minimize(NLSE, params, exp_from = 400, output_runtime=False)
    #MSE calc
    #MSE_NLSE = sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.data[-1, 0, :-1])**2) / len(X_grid[:,0])
    #print(MSE_NLSE, 'using sum')
    MSE_NLSE = np.sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.data[-1, 0, :-1])**2) / len(X_grid[:,0])
    MSE_NLSE = np.float64(MSE_NLSE)
    print('\nNLSE DONE\n')
    ##################################################################################################

    #Finally the MSE_global is the accumulated MSE between the three problems
    MSE_global = [MSE_Oscilador, MSE_Well, MSE_NLSE]
    print(f'MSE_Oscilador: {round(MSE_Oscilador, 5)}\nMSE_Well: {round(MSE_Well, 5)}\nMSE_NLSE: {round(MSE_NLSE, 5)}\n\n')
    #Contador para saber cuántas veces se emplea cada problema
    # Find the index of the max MSE
    max_index = np.argmax(MSE_global)  # Gets the index of the max value
    if max_index==0:
        global timesQOsc
        timesQOsc+=1
    if max_index==1:
        global timesWell
        timesWell+=1
    if max_index==2:
        global timesNLSE
        timesNLSE+=1
    #rmse_values = np.random.uniform(0.1, 1.0, 3)  # Replace with real RMSE computations
    return np.max(MSE_global) #max(rmse_values)  # Min-Max approach: minimize worst RMSE

global call
call = 0

# Wrapper function for Bayesian Optimization
@use_named_args(search_space)
def objective(**params):
    # # Ensure categorical values are correctly interpreted as integers. Categorical was failing when GP takes action because cannot handle categorical GPT suggested this
    # params["size_gen"] = int(params["size_gen"])
    # params["dead_iter"] = int(params["dead_iter"])
    # params["init_kernels"] = int(params["init_kernels"])
    params['alpha'] = max(0.02, min(0.4, params['alpha']))  
    params['elite_factor'] = max(0.02, min(0.4, params['elite_factor']))  
    params['std_lower_bound'] = max(0.01, min(0.2, params['std_lower_bound']))
    global call
    call +=1
    print(f'\n##############################################\nBO call number {call} of 50\n##############################################\n\n')
    return evaluate_algorithm(**params)

BO_params425 = {}

# Run Bayesian Optimization
print(f'\n##############################################\nBO for {n_variables[1]} variables\n##############################################\n\n')
result = gp_minimize(
    func=objective,      # Objective function
    dimensions=search_space,  # Search space
    n_calls=50,         # Number of function evaluations
    random_state=42,     # For reproducibility
    n_initial_points=12
)

# Best parameters found
best_params = dict(zip(["size_gen", "dead_iter", "alpha", "elite_factor", "std_lower_bound", "init_kernels"], result.x))
BO_params425['params'] = best_params
BO_params425['problem instances counts'] = f'MSE de Oscilador fue usado {timesQOsc} veces. MSE de Pozo Infinito fue usado {timesWell} veces. MSE de NLSE fue usado {timesNLSE} veces'
print(f'\n##############################################\nBO of EGPA parameters ended\n##############################################\n\n')
print(f"Best EGPA Parameters for {n_variables[1]}:", best_params)
print(f'\nContador\n\nMSE de Oscilador fue usado {timesQOsc} veces\nMSE de Pozo Infinito fue usado {timesWell} veces\nMSE de NLSE fue usado {timesNLSE} veces\n')
print("Best MSE (Min-Max Approach):", result.fun)

import pickle
ruta = 'BO_params/425EGPA'
with open(f'{ruta}.pkl', "wb") as file:
    pickle.dump(BO_params425, file)


for keys in BO_params425.keys():
    print(f'Problema {keys}: {BO_params425[keys]}\n')



##############################################
BO for 425 variables
##############################################



##############################################
BO call number 1 of 50
##############################################


Quantum Harmonic Oscillator IN PROGRESS

F should be approx -0.082
	NFEVALS = 67150 F = [-0.08204784]
	X = [-4.44773562e-09  3.19309116e-04  6.03263239e-04  8.53652015e-04
  1.07281120e-03  1.26250506e-03  1.42493275e-03  1.56192889e-03
  1.67563041e-03  1.76792267e-03  1.84073155e-03  1.89595915e-03
  1.93544671e-03  1.96112332e-03  1.97472083e-03  1.97791967e-03
  1.97254423e-03  1.96026723e-03  1.94270280e-03  1.92140025e-03
  1.89794623e-03  1.87374641e-03  1.85038367e-03  1.82914949e-03
  1.81137212e-03  1.79840694e-03  1.79147451e-03  1.79177167e-03
  1.80043550e-03  1.81852214e-03  1.84723981e-03  1.88746128e-03
  1.94022046e-03  2.00638114e-03  2.08686010e-03  2.18252910e-03
  2.29412125e-03  2.42249020e-03  2.56836462e-03  2.73236032e-03
  2.9

In [1]:
from EGPA import *
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Add the main directory to sys.path
sys.path.append(main_dir)
from OptFunctions.oneDQHarmOscillatorfunctions import *
from OptFunctions.NonLinearSE import *
from OptFunctions.oneDwellfunctions import *
#from utils import _parallel_apply_along_axis
import time
import warnings
warnings.filterwarnings("ignore")

#np.int = int
from skopt import gp_minimize
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args

n_variables = [50, 425, 800] #Tamaños del grid (número de dimensiones del problema de optimización)

# Optimización de parámetros de EGPA
# Define the search space
#¿¿?¿?¿?¿?¿?¿?¿?¿? Categorical([10, 30, 60, 100], name="dead_iter") does not work becaues GP cannot handle categorical values
# search_space = [
#     Integer(50, 1000, name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
#     Integer(10, 100, name="dead_iter"),   # Discrete values Categorical([10, 30, 60, 100], name="dead_iter") was not working because GP cannot handle categorical!!!!
#     Real(0.02, 0.4, name="alpha", prior='log-uniform'),                # Continuous values
#     Real(0.02, 0.4, name="elite_factor", prior='log-uniform'),        # Continuous values
#     Real(0.01, 0.2, name="std_lower_bound", prior='log-uniform'),     # Continuous values
#     Integer(2, 10, name="init_kernels")      # Discrete values
# ]

search_space = [
    Categorical([50, 150, 350, 550, 850, 1000], name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
    Categorical([10, 30, 60, 100], name="dead_iter"),   # Discrete values Categorical([10, 30, 60, 100], name="dead_iter") was not working because GP cannot handle categorical!!!!
    Real(0.02, 0.4, name="alpha", prior='log-uniform'),                # Continuous values
    Real(0.02, 0.4, name="elite_factor", prior='log-uniform'),        # Continuous values
    Real(0.01, 0.2, name="std_lower_bound", prior='log-uniform'),     # Continuous values
    Integer(2, 10, name="init_kernels")      # Discrete values
]

global timesQOSc
timesQOsc = 0
global timesWell
timesWell = 0
global timesNLSE
timesNLSE = 0

# Simulated function that runs the optimization and computes RMSE
def evaluate_algorithm(size_gen, dead_iter, alpha, elite_factor, std_lower_bound, init_kernels):
    """
    This function should run the actual optimization algorithm using the given parameters.
    It should return the worst MSE (max MSE across the three optimization problems).
    
    Replace the `np.random.uniform()` call with the actual function evaluation.
    """
    ##########################################################################################
    ##########################################################################################
    ########################### OSCILADOR ARMÓNICO ###########################################
    ##########################################################################################
    print('Quantum Harmonic Oscillator IN PROGRESS\n')
    X_grid = np.linspace(start=-4, stop=4, num=n_variables[2]).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=-4 (-inf) x=4 (inf) que son 0
    X_train = X_grid[[0,-1]]
    Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array

    #Analytical solutions definitions
    #Ground state
    APsi0 = np.exp(-X_grid**2/2)
    integral =0
    for i in range(len(APsi0)):
        integral = integral + (APsi0[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi0)-1))

    B = (1/integral)**0.5
    APsi0 = B * APsi0

    #1st excitated state
    APsi1 = np.exp(-X_grid**2/2)*2*X_grid
    integral =0
    for i in range(len(APsi1)):
        integral = integral + (APsi1[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi1)-1))

    B = (1/integral)**0.5
    APsi1 = -1 * B * APsi1 #signo menos para correspondencia con solución algoritmo (simetrización)

    #2nd excitated state
    APsi2 = np.exp(-X_grid**2/2)*(4*X_grid**2-2)
    integral =0
    for i in range(len(APsi2)):
        integral = integral + (APsi2[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi2)-1))

    B = (1/integral)**0.5
    APsi2 = B * APsi2

    #3rd excitated state
    APsi3 = np.exp(-X_grid**2/2)*(8*X_grid**3-12*X_grid)
    integral =0
    for i in range(len(APsi3)):
        integral = integral + (APsi3[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi3)-1))

    B = (1/integral)**0.5
    APsi3 = -1 * B * APsi3 #signo menos para correspondencia con solución algoritmo (simetrización)

    APsis = {}
    APsis['APsi0'] = APsi0[:, 0]
    APsis['APsi1'] = APsi1[:, 0]
    APsis['APsi2'] = APsi2[:, 0]
    APsis['APsi3'] = APsi3[:, 0]

    ### Ahora el código con EGPA que calcula los 4 primeros estados excitados del problema oscilador armónico
    params = {'x':X_grid}
    #Ground state
    print(f'F should be approx -0.082')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHOground = Gaussian.minimize(oneDHOExpE, params, exp_from = 3, output_runtime=False)
    #Se guardan resultados de algoritmo para siguiente estado excitado
    Psi_ground=egparesultQHOground.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E0 = np.sqrt(-np.log(-egparesultQHOground.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así 
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground}
    print(f'\n  |Psi0> done\n')

    #1er estado excitado
    print(f'F should be approx -4.54e-5')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHO1 = Gaussian.minimize(oneDHOExpE1, params, exp_from = 5, output_runtime=False)
    Psi1=egparesultQHO1.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E1 = np.sqrt(-np.log(-egparesultQHO1.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1}
    print(f'\n  |Psi1> done\n')

    #2o estado excitado
    print(f'F should be approx -4.54e-5')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHO2 = Gaussian.minimize(oneDHOExpE2, params, exp_from = 5, output_runtime=False)
    Psi2=egparesultQHO2.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E2 = np.sqrt(-np.log(-egparesultQHO2.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2 }
    print(f'\n  |Psi2> done\n')

    #3er estado excitado
    print(f'F should be approx -4.54e-5')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels, std_lower_bound=std_lower_bound, parallelize=True, init_data=(X_train, Y_train))
    egparesultQHO3 = Gaussian.minimize(oneDHOExpE3, params, exp_from = 5, output_runtime=False)
    Psi3=egparesultQHO3.data[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E3 = np.sqrt(-np.log(-egparesultQHO3.data[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2, 'E3': E3, 'Psi3': Psi3 }
    print(f'\n  |Psi3> done\n')

    #Cálculos de métricas error MSE de cada estado con estado teórico
    MSEs = []
    for i in range(4):
        mse = np.mean((params[f'Psi{i}'] - APsis[f'APsi{i}'])**2)
        MSEs.append(mse)

    #MSE_Oscilador = np.sum(MSEs)
    MSE_Oscilador = np.mean(MSEs)
    print('\nQuantum Harmonic Oscillator DONE\n')

    ######################################################################################
    ######################################################################################
    ############## POZO DE POTENCIAL INFINITO 1D WELL ####################################
    ######################################################################################
    print('\nInfinite Well IN PROGRESS\n')
    L=5
    X_grid = np.linspace(start=0, stop=L, num=n_variables[2]).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=0 x=5 que son 0
    X_train = X_grid[[0,-1]]
    Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array
    params = {'L':L}

    #Ground state n=1
    print(f'F should be approx -0.9618')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels ,parallelize=True, init_data=(X_train, Y_train))
    egparesultgroundWell = Gaussian.minimize(oneD_wellExpE, params, exp_from = 2, output_runtime=False)
    Psi_fund=egparesultgroundWell.data[-1, 0, :-1]
    E_fund = np.sqrt(-np.log(-egparesultgroundWell.data[-1,0,-1])) # def exponencial de función de coste la energía se determina a partir del fitness así.
    #En este caso no hay constante multiplicando exponente exponencial
    params = {'L':L, 'E': E_fund, 'Psi': Psi_fund}
    print(f'\n  |Psi0> done\n')

    #1excitated state n=2
    print(f'F should be approx -0.7042')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
    egparesultWellexc1 = Gaussian.minimize(oneD_wellExpEexcitated, params, exp_from = 4, output_runtime=False)
    Psi_1=egparesultWellexc1.data[-1, 0, :-1]
    E1 = np.sqrt(-np.log(-egparesultWellexc1.data[-1,0,-1]))
    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1}
    print(f'\n  |Psi1> done\n')

    #2nd excitated state n=3
    print(f'F should be approx -0.3775')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
    egparesultWellexc2 = Gaussian.minimize(oneD_wellExpEexcitated3, params, exp_from = 5, output_runtime=False)
    Psi_2=egparesultWellexc2.data[-1, 0, :-1]
    E2 = np.sqrt(-np.log(-egparesultWellexc2.data[-1,0,-1]))
    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2}
    print(f'\n  |Psi2> done\n')

    #3rd excitated state n=4
    print(f'F should be approx -0.1482')
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels,parallelize=True, init_data=(X_train, Y_train))
    egparesultWellexc3 = Gaussian.minimize(oneD_wellExpEexcitated4, params, exp_from = 9, output_runtime=False)
    Psi_3 = egparesultWellexc3.data[-1, 0, :-1]
    E3 = np.sqrt(-np.log(-egparesultWellexc3.data[-1,0,-1]))
    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2, 'E3':E3, 'Psi3': Psi_3}
    print(f'\n  |Psi3> done\n')

    #Soluciones teóricas
    ### Soluciones analíticas
    # def Psi_n(n):
    #     return((2/L)**0.5*np.sin(n*np.pi*np.linspace(0, L, n_variables)/L))

    APsisWell = {}
    for n in range(4):
        APsisWell[f'APsi{n}'] = (2/L)**0.5*np.sin((n+1)*np.pi*np.linspace(0, L, n_variables[2])/L)


    #Cálculos de métricas error MSE de cada estado con estado teórico
    MSEs = []
    for i in range(4):
        mse = np.mean((params[f'Psi{i}'] - APsisWell[f'APsi{i}'])**2)
        MSEs.append(mse)

    #MSE_Well = np.sum(MSEs)
    #Mean is more correct because sum will make the MSE in NLSE problem probably always smaller 
    MSE_Well = np.mean(MSEs)

    print('\nInfinite Well DONE\n')

    ######################################################################################
    ######################################################################################
    #################### N L S E Non linear Schroedinger Equation ########################
    ######################################################################################
    print('\nNLSE IN PROGRESS\n')
    L=2
    #El grid se define en para los tres problemas de optimización con el mismo número de variables
    X_grid = np.linspace(start=-1, stop=1, num=n_variables[2]+1).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=0 que son 1
    #X_train = X_grid[n_variables//2]
    X_train = np.array((0)).reshape(-1,1)
    Y_train = np.linspace(start=1, stop=1, num=1).reshape(-1,1) #Para que sea 2D array
    params = {'L':L}
    Gaussian = EGPA(size_gen=size_gen, grid_X=X_grid, max_iter=220, dead_iter=dead_iter, alpha=alpha, elite_factor=elite_factor, init_kernels=init_kernels ,parallelize=True, init_data=(X_train, Y_train), normalize_solutions=False, symmetrize_solutions=False)
    egparesultNLSE = Gaussian.minimize(NLSE, params, exp_from = 400, output_runtime=False)
    #MSE calc
    #MSE_NLSE = sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.data[-1, 0, :-1])**2) / len(X_grid[:,0])
    #print(MSE_NLSE, 'using sum')
    MSE_NLSE = np.sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.data[-1, 0, :-1])**2) / len(X_grid[:,0])
    MSE_NLSE = np.float64(MSE_NLSE)
    print('\nNLSE DONE\n')
    ##################################################################################################

    #Finally the MSE_global is the accumulated MSE between the three problems
    MSE_global = [MSE_Oscilador, MSE_Well, MSE_NLSE]
    print(f'MSE_Oscilador: {round(MSE_Oscilador, 5)}\nMSE_Well: {round(MSE_Well, 5)}\nMSE_NLSE: {round(MSE_NLSE, 5)}\n\n')
    #Contador para saber cuántas veces se emplea cada problema
    # Find the index of the max MSE
    max_index = np.argmax(MSE_global)  # Gets the index of the max value
    if max_index==0:
        global timesQOsc
        timesQOsc+=1
    if max_index==1:
        global timesWell
        timesWell+=1
    if max_index==2:
        global timesNLSE
        timesNLSE+=1
    #rmse_values = np.random.uniform(0.1, 1.0, 3)  # Replace with real RMSE computations
    return np.max(MSE_global) #max(rmse_values)  # Min-Max approach: minimize worst RMSE

global call
call = 0

# Wrapper function for Bayesian Optimization
@use_named_args(search_space)
def objective(**params):
    # # Ensure categorical values are correctly interpreted as integers. Categorical was failing when GP takes action because cannot handle categorical GPT suggested this
    # params["size_gen"] = int(params["size_gen"])
    # params["dead_iter"] = int(params["dead_iter"])
    # params["init_kernels"] = int(params["init_kernels"])
    params['alpha'] = max(0.02, min(0.4, params['alpha']))  
    params['elite_factor'] = max(0.02, min(0.4, params['elite_factor']))  
    params['std_lower_bound'] = max(0.01, min(0.2, params['std_lower_bound']))
    global call
    call +=1
    print(f'\n##############################################\nBO call number {call} of 50\n##############################################\n\n')
    return evaluate_algorithm(**params)

BO_params800 = {}

# Run Bayesian Optimization
print(f'\n##############################################\nBO for {n_variables[2]} variables\n##############################################\n\n')
result = gp_minimize(
    func=objective,      # Objective function
    dimensions=search_space,  # Search space
    n_calls=50,         # Number of function evaluations
    random_state=42,     # For reproducibility
    n_initial_points=12
)

# Best parameters found
best_params = dict(zip(["size_gen", "dead_iter", "alpha", "elite_factor", "std_lower_bound", "init_kernels"], result.x))
BO_params800['params'] = best_params
BO_params800['problem instances counts'] = f'MSE de Oscilador fue usado {timesQOsc} veces. MSE de Pozo Infinito fue usado {timesWell} veces. MSE de NLSE fue usado {timesNLSE} veces'
print(f'\n##############################################\nBO of EGPA parameters ended\n##############################################\n\n')
print(f"Best EGPA Parameters for {n_variables[2]}:", best_params)
print(f'\nContador\n\nMSE de Oscilador fue usado {timesQOsc} veces\nMSE de Pozo Infinito fue usado {timesWell} veces\nMSE de NLSE fue usado {timesNLSE} veces\n')
print("Best MSE (Min-Max Approach):", result.fun)

import pickle
ruta = 'BO_params/800EGPA'
with open(f'{ruta}.pkl', "wb") as file:
    pickle.dump(BO_params800, file)


for keys in BO_params800.keys():
    print(f'Problema {keys}: {BO_params800[keys]}\n')



##############################################
BO for 800 variables
##############################################



##############################################
BO call number 1 of 50
##############################################


Quantum Harmonic Oscillator IN PROGRESS

F should be approx -0.082
	NFEVALS = 82450 F = [-0.08204142]
	X = [-1.42348683e-08 -1.18975206e-04 -2.30492432e-04 -3.34689037e-04
 -4.31727177e-04 -5.21841361e-04 -6.05189364e-04 -6.81943583e-04
 -7.52295798e-04 -8.16361547e-04 -8.74335767e-04 -9.26403090e-04
 -9.72697895e-04 -1.01337343e-03 -1.04861122e-03 -1.07848058e-03
 -1.10329427e-03 -1.12298629e-03 -1.13782300e-03 -1.14788307e-03
 -1.15333854e-03 -1.15430262e-03 -1.15091467e-03 -1.14328069e-03
 -1.13146468e-03 -1.11566679e-03 -1.09595440e-03 -1.07241312e-03
 -1.04510004e-03 -1.01422424e-03 -9.79808615e-04 -9.41950936e-04
 -9.00788355e-04 -8.56270634e-04 -8.08626319e-04 -7.57789225e-04
 -7.03901357e-04 -6.47037030e-04 -5.87242725e-04 -5.24525805e-04
 -4.5